# SCM 用 Data Agent ベストプラクティス検証データの生成

ワークスペース **`Book`** の Lakehouse `lh_scm_demo` に、書籍『SCM データで体験する Data Agent ベストプラクティス検証』のための サプライチェーン サンプルデータを生成します。

## 生成内容

**A. ベストプラクティス未適用デモ用（悪命名）**
- `tbl_001`〜`tbl_005`（列名も `col_a, col_b, ...`）

**B. ベストプラクティス適用版（良命名）**

| 種別 | テーブル | 行数目安 |
|------|---------|---------|
| Dim | `dim_supplier` | 60 |
| Dim | `dim_material` | 200 |
| Dim | `dim_product` | 80 |
| Dim | `dim_plant` | 12 |
| Dim | `dim_warehouse` | 24 |
| Dim | `dim_date` | 1,461 |
| Fact | `fact_purchase_orders` | 18,000 |
| Fact | `fact_goods_receipts` | 17,200 |
| Fact | `fact_inventory_snapshot` | 〜90,000 |
| Fact | `fact_production_orders` | 8,000 |
| Fact | `fact_shipments` | 14,000 |
| Fact | `fact_quality_inspections` | 5,000 |

## 前提
Fabric Notebook で実行し、Lakehouse `lh_scm_demo` をデフォルトとしてアタッチしてください。

In [ ]:
import random
from datetime import date, timedelta
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, DateType, BooleanType
)

random.seed(42)
print(f"Spark version: {spark.version}")
print("デフォルト Lakehouse `lh_scm_demo` がアタッチされていることを確認してください")

## 1. ディメンションテーブルの生成

In [ ]:
# ───── dim_supplier ─────
regions = ["日本", "中国", "台湾", "韓国", "ベトナム", "タイ", "ドイツ", "米国"]
grades = ["A", "B", "C"]
suppliers = []
for i in range(1, 61):
    suppliers.append((
        f"S-{i:04d}",
        f"サプライヤー_{i:03d}",
        random.choice(regions),
        random.choice(grades),
        random.randint(7, 60),                # standard_lead_time_days
        date(random.randint(2010, 2023), random.randint(1,12), random.randint(1,28)),
    ))
schema = StructType([
    StructField("supplier_id", StringType(), False),
    StructField("supplier_name", StringType(), False),
    StructField("country", StringType(), False),
    StructField("supplier_grade", StringType(), False),
    StructField("standard_lead_time_days", IntegerType(), False),
    StructField("contract_start_date", DateType(), False),
])
spark.createDataFrame(suppliers, schema).write.mode("overwrite").format("delta").saveAsTable("dim_supplier")
print("dim_supplier:", len(suppliers))

In [ ]:
# ───── dim_material ─────
categories = ["金属", "樹脂", "電子部品", "化学品", "梱包材"]
subcat_map = {
    "金属": ["ステンレス板", "アルミ材", "銅線"],
    "樹脂": ["ABS樹脂", "PP樹脂", "PET樹脂"],
    "電子部品": ["抵抗", "コンデンサ", "ICチップ"],
    "化学品": ["塗料", "接着剤", "溶剤"],
    "梱包材": ["段ボール", "緩衝材", "ラベル"],
}
materials = []
supplier_ids = [f"S-{i:04d}" for i in range(1, 61)]
for i in range(1, 201):
    cat = random.choice(categories)
    sub = random.choice(subcat_map[cat])
    materials.append((
        f"M-{i:04d}",
        f"{sub}_{i:03d}",
        cat,
        sub,
        "RM",                                 # material_type
        random.choice(supplier_ids),         # primary_supplier_id
        random.choice([10, 50, 100, 200, 500]),       # min_order_qty (MOQ)
        random.choice([20, 50, 100, 200, 500, 1000]), # safety_stock_qty (SS)
        round(random.uniform(50, 5000), 2),   # unit_cost_excl_tax (JPY)
    ))
schema = StructType([
    StructField("material_id", StringType(), False),
    StructField("material_name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("sub_category", StringType(), False),
    StructField("material_type", StringType(), False),
    StructField("primary_supplier_id", StringType(), False),
    StructField("min_order_qty", IntegerType(), False),
    StructField("safety_stock_qty", IntegerType(), False),
    StructField("unit_cost_excl_tax", DoubleType(), False),
])
spark.createDataFrame(materials, schema).write.mode("overwrite").format("delta").saveAsTable("dim_material")
print("dim_material:", len(materials))

In [ ]:
# ───── dim_product (完成品 FG) ─────
families = ["産業機器", "家電", "自動車部品", "医療機器"]
products = []
for i in range(1, 81):
    fam = random.choice(families)
    products.append((
        f"FG-{i:04d}",
        f"{fam}製品_{i:03d}",
        fam,
        "FG",
        round(random.uniform(5000, 200000), 2),  # unit_price_excl_tax
        random.choice(["主力", "通常", "型落ち"]),
    ))
schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("product_family", StringType(), False),
    StructField("product_type", StringType(), False),
    StructField("unit_price_excl_tax", DoubleType(), False),
    StructField("product_status", StringType(), False),
])
spark.createDataFrame(products, schema).write.mode("overwrite").format("delta").saveAsTable("dim_product")
print("dim_product:", len(products))

In [ ]:
# ───── dim_plant (工場) ─────
plants = []
for i in range(1, 13):
    plants.append((
        f"P-{i:02d}",
        f"第{i}工場",
        random.choice(["日本", "中国", "タイ", "米国"]),
        random.choice(["組立", "加工", "成形"]),
    ))
schema = StructType([
    StructField("plant_id", StringType(), False),
    StructField("plant_name", StringType(), False),
    StructField("country", StringType(), False),
    StructField("plant_type", StringType(), False),
])
# plant_code = plant_id を別名で保持
from pyspark.sql.functions import col
df = spark.createDataFrame(plants, schema).withColumn("plant_code", col("plant_id"))
df.write.mode("overwrite").format("delta").saveAsTable("dim_plant")
print("dim_plant:", len(plants))

In [ ]:
# ───── dim_warehouse (倉庫) ─────
warehouses = []
for i in range(1, 25):
    warehouses.append((
        f"W-{i:02d}",
        f"倉庫_{i:02d}",
        random.choice(["北日本", "東日本", "西日本", "海外"]),
        random.choice(["完成品", "原材料", "両用"]),
    ))
schema = StructType([
    StructField("warehouse_id", StringType(), False),
    StructField("warehouse_name", StringType(), False),
    StructField("region", StringType(), False),
    StructField("warehouse_type", StringType(), False),
])
spark.createDataFrame(warehouses, schema).write.mode("overwrite").format("delta").saveAsTable("dim_warehouse")
print("dim_warehouse:", len(warehouses))

In [ ]:
# ───── dim_date ─────
rows = []
d = date(2023, 1, 1)
end = date(2026, 12, 31)
while d <= end:
    q = (d.month - 1) // 3 + 1
    rows.append((
        int(d.strftime("%Y%m%d")),
        d, d.year, q, d.month, d.day,
        f"{d.year}-Q{q}", f"{d.year}-{d.month:02d}",
        d.weekday() >= 5,
    ))
    d += timedelta(days=1)
schema = StructType([
    StructField("date_key", IntegerType(), False),
    StructField("date", DateType(), False),
    StructField("year", IntegerType(), False),
    StructField("quarter", IntegerType(), False),
    StructField("month", IntegerType(), False),
    StructField("day", IntegerType(), False),
    StructField("year_quarter", StringType(), False),
    StructField("year_month", StringType(), False),
    StructField("is_weekend", BooleanType(), False),
])
spark.createDataFrame(rows, schema).write.mode("overwrite").format("delta").saveAsTable("dim_date")
print("dim_date:", len(rows))

## 2. ファクトテーブルの生成

In [ ]:
# ───── fact_purchase_orders + fact_goods_receipts ─────
po_rows = []
gr_rows = []
material_supplier = {m[0]: m[5] for m in materials}
material_cost     = {m[0]: m[8] for m in materials}
supplier_lt       = {s[0]: s[4] for s in suppliers}
supplier_grade    = {s[0]: s[3] for s in suppliers}

po_id = 1
gr_id = 1
start = date(2024, 1, 1)
for _ in range(18000):
    mat = random.choice(materials)
    mid = mat[0]
    sid = material_supplier[mid]
    po_date = start + timedelta(days=random.randint(0, 850))
    lt = supplier_lt[sid]
    promised = po_date + timedelta(days=lt)
    qty = random.choice([50, 100, 200, 500, 1000])
    status = random.choices(["CLOSED", "OPEN", "CANCELLED"], weights=[85, 12, 3])[0]
    po_rows.append((
        f"PO-{po_id:07d}", sid, mid, po_date, promised, qty,
        round(material_cost[mid], 2), status,
    ))
    # 入荷生成（CLOSED の 96% で生成）
    if status == "CLOSED" and random.random() < 0.96:
        # サプライヤー等級ごとに遅延傾向
        grade = supplier_grade[sid]
        delay_mu = {"A": -1, "B": 2, "C": 6}[grade]
        actual = promised + timedelta(days=int(random.gauss(delay_mu, 3)))
        recv_qty = qty if random.random() < 0.92 else int(qty * random.uniform(0.7, 0.99))
        gr_rows.append((
            f"GR-{gr_id:07d}", f"PO-{po_id:07d}", actual, recv_qty,
        ))
        gr_id += 1
    po_id += 1

schema_po = StructType([
    StructField("po_id", StringType(), False),
    StructField("supplier_id", StringType(), False),
    StructField("material_id", StringType(), False),
    StructField("po_date", DateType(), False),
    StructField("promised_date", DateType(), False),
    StructField("ordered_qty", IntegerType(), False),
    StructField("unit_price_excl_tax", DoubleType(), False),
    StructField("po_status", StringType(), False),
])
spark.createDataFrame(po_rows, schema_po).write.mode("overwrite").format("delta").saveAsTable("fact_purchase_orders")

schema_gr = StructType([
    StructField("gr_id", StringType(), False),
    StructField("po_id", StringType(), False),
    StructField("actual_receipt_date", DateType(), False),
    StructField("received_qty", IntegerType(), False),
])
spark.createDataFrame(gr_rows, schema_gr).write.mode("overwrite").format("delta").saveAsTable("fact_goods_receipts")
print("fact_purchase_orders:", len(po_rows), "  fact_goods_receipts:", len(gr_rows))

In [ ]:
# ───── fact_production_orders ─────
prod_rows = []
plant_ids = [p[0] for p in plants]
for i in range(1, 8001):
    p = random.choice(products)
    plan_start = start + timedelta(days=random.randint(0, 850))
    plan_end = plan_start + timedelta(days=random.randint(1, 14))
    delay = max(0, int(random.gauss(0.5, 2)))
    actual_end = plan_end + timedelta(days=delay)
    qty = random.choice([10, 50, 100, 200])
    prod_rows.append((
        f"PRD-{i:06d}",
        random.choice(plant_ids),
        p[0],
        plan_start, plan_end, actual_end,
        qty,
        random.choices(["COMPLETED","IN_PROGRESS","CANCELLED"], weights=[88,9,3])[0],
    ))
schema = StructType([
    StructField("production_order_id", StringType(), False),
    StructField("plant_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("planned_start_date", DateType(), False),
    StructField("planned_end_date", DateType(), False),
    StructField("actual_end_date", DateType(), False),
    StructField("produced_qty", IntegerType(), False),
    StructField("status", StringType(), False),
])
spark.createDataFrame(prod_rows, schema).write.mode("overwrite").format("delta").saveAsTable("fact_production_orders")
print("fact_production_orders:", len(prod_rows))

In [ ]:
# ───── fact_quality_inspections ─────
qc_rows = []
for i in range(1, 5001):
    p = random.choice(products)
    plant = random.choice(plant_ids)
    insp_date = start + timedelta(days=random.randint(0, 850))
    inspected = random.choice([50, 100, 200, 500])
    defect_rate = random.choices([0.001, 0.005, 0.01, 0.03, 0.08], weights=[40,30,15,10,5])[0]
    defect = max(0, int(inspected * random.uniform(defect_rate*0.5, defect_rate*1.5)))
    qc_rows.append((
        f"QC-{i:06d}", plant, p[0], insp_date, inspected, defect,
    ))
schema = StructType([
    StructField("inspection_id", StringType(), False),
    StructField("plant_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("inspection_date", DateType(), False),
    StructField("inspected_qty", IntegerType(), False),
    StructField("defect_qty", IntegerType(), False),
])
spark.createDataFrame(qc_rows, schema).write.mode("overwrite").format("delta").saveAsTable("fact_quality_inspections")
print("fact_quality_inspections:", len(qc_rows))

In [ ]:
# ───── fact_shipments ─────
ship_rows = []
wh_ids = [w[0] for w in warehouses]
regions_ship = ["北日本", "東日本", "西日本", "海外"]
for i in range(1, 14001):
    p = random.choice(products)
    sd = start + timedelta(days=random.randint(0, 850))
    planned = random.choice([10, 20, 50, 100])
    is_full = random.random() < 0.92
    is_on = random.random() < 0.88
    shipped = planned if is_full else int(planned * random.uniform(0.6, 0.95))
    ship_rows.append((
        f"SHP-{i:06d}",
        random.choice(wh_ids),
        p[0],
        sd,
        planned,
        shipped,
        bool(is_on),
        bool(is_full),
        random.choice(regions_ship),
    ))
schema = StructType([
    StructField("shipment_id", StringType(), False),
    StructField("warehouse_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("shipment_date", DateType(), False),
    StructField("planned_qty", IntegerType(), False),
    StructField("shipped_qty", IntegerType(), False),
    StructField("is_on_time", BooleanType(), False),
    StructField("is_in_full", BooleanType(), False),
    StructField("customer_region", StringType(), False),
])
spark.createDataFrame(ship_rows, schema).write.mode("overwrite").format("delta").saveAsTable("fact_shipments")
print("fact_shipments:", len(ship_rows))

In [ ]:
# ───── fact_inventory_snapshot (月末スナップショット) ─────
inv_rows = []
ss_map = {m[0]: m[7] for m in materials}
# 24 months × 24 wh × 〜200 mat（サンプリング）
month_starts = []
y = 2024
for m in range(1, 13):
    month_starts.append(date(y, m, 1))
for m in range(1, 13):
    month_starts.append(date(y+1, m, 1))
month_starts.append(date(2026, 1, 1))
month_starts.append(date(2026, 2, 1))
month_starts.append(date(2026, 3, 1))
month_starts.append(date(2026, 4, 1))

for snap in month_starts:
    snap_eom = (snap.replace(day=28) + timedelta(days=4)).replace(day=1) - timedelta(days=1)
    snap_key = int(snap_eom.strftime("%Y%m%d"))
    for w in wh_ids:
        # 各倉庫で部材を 150 件サンプリング
        for mat in random.sample(materials, 150):
            opening = random.randint(0, 3000)
            closing = max(0, opening + random.randint(-500, 500))
            ss = ss_map[mat[0]]
            is_stockout = closing == 0
            inv_rows.append((
                snap_key, w, mat[0], opening, closing,
                random.randint(0, 1000), random.randint(0, 1000),
                bool(is_stockout), bool(closing < ss),
            ))

schema = StructType([
    StructField("snapshot_date_key", IntegerType(), False),
    StructField("warehouse_id", StringType(), False),
    StructField("material_id", StringType(), False),
    StructField("opening_qty", IntegerType(), False),
    StructField("closing_qty", IntegerType(), False),
    StructField("inbound_qty", IntegerType(), False),
    StructField("outbound_qty", IntegerType(), False),
    StructField("is_stockout", BooleanType(), False),
    StructField("is_below_ss", BooleanType(), False),
])
spark.createDataFrame(inv_rows, schema).write.mode("overwrite").format("delta").saveAsTable("fact_inventory_snapshot")
print("fact_inventory_snapshot:", len(inv_rows))

## 3. 悪命名テーブル（ベストプラクティス #1 デモ用）

意味のない名前 (`tbl_001`〜`tbl_005`、列 `col_a`, `col_b`, ...) で同等データを別名保存します。

In [ ]:
from pyspark.sql import functions as F

def rename_to_cols(df):
    new = [f"col_{chr(97+i)}" for i in range(len(df.columns))]
    for old, n in zip(df.columns, new):
        df = df.withColumnRenamed(old, n)
    return df

src_map = {
    "tbl_001": "fact_purchase_orders",
    "tbl_002": "fact_goods_receipts",
    "tbl_003": "fact_inventory_snapshot",
    "tbl_004": "fact_shipments",
    "tbl_005": "dim_supplier",
}
for tgt, src in src_map.items():
    df = spark.read.table(src)
    rename_to_cols(df).write.mode("overwrite").format("delta").saveAsTable(tgt)
    print(f"{tgt} <- {src}")

## 4. 評価用 30 問データセットの保存

`evaluate_scm_dataagents.ipynb` から読み込めるよう CSV として書き出します。

In [ ]:
import pandas as pd

questions = [
    (1,  "入荷",       "2026年3月のオンタイム入荷率は？"),
    (2,  "入荷",       "サプライヤー S-0007 の直近90日の中央リードタイム日数は？"),
    (3,  "入荷",       "リードタイムが平均より20%以上長いサプライヤートップ5は？"),
    (4,  "発注",       "直近月の発注金額が大きい部材カテゴリ上位3は？"),
    (5,  "発注",       "材料名に「ステンレス」を含む部材の今月発注金額は？"),
    (6,  "在庫",       "倉庫W-08で安全在庫を割っている部材は？"),
    (7,  "在庫",       "完成品 FG-0104 の今日時点の DOI は？"),
    (8,  "在庫",       "DOI が 60 日を超える完成品上位10は？"),
    (9,  "在庫",       "直近月の欠品率が最も高い倉庫は？"),
    (10, "在庫",       "倉庫別×部材カテゴリ別の欠品率トップ10は？"),
    (11, "出荷",       "4月の OTIF は？"),
    (12, "出荷",       "直近6か月の OTIF 月次推移を教えて"),
    (13, "出荷",       "OTIF が最も低い顧客地域は？"),
    (14, "出荷",       "直近月の出荷金額トップ5の完成品は？"),
    (15, "生産",       "4月の工場 P-03 の不良率は？"),
    (16, "生産",       "不良率が前月比で悪化した工場は？"),
    (17, "生産",       "製品ファミリ別の生産遅延件数（直近月）は？"),
    (18, "生産",       "生産遅延が3件以上ある完成品は？"),
    (19, "横断",       "サプライヤー遅延と完成品の生産遅延の相関は？"),
    (20, "横断",       "不良率が高い完成品に共通する部材サプライヤーは？"),
    (21, "用語",       "OTD と OTIF の違いを定義どおりに説明して"),
    (22, "用語",       "DOI を定義どおりに計算して、上位5の完成品を返して"),
    (23, "用語",       "SS を切った部材の数を月別に教えて"),
    (24, "フォールバック", "2030年の OTIF を教えて"),
    (25, "フォールバック", "サプライヤー S-9999 のリードタイムは？"),
    (26, "スタイル",    "倉庫W-08の在庫健全性を判定して"),
    (27, "スタイル",    "サプライヤー S-0007 の評価を一文で"),
    (28, "同質性",      "2026-03 の OTIF は？"),
    (29, "同質性",      "DOI 上位5を返して"),
    (30, "同質性",      "不良率トレンド月次"),
]
qdf = pd.DataFrame(questions, columns=["id", "category", "question"])
qdf["expected_answer"] = ""  # 各環境で実データから埋める
qdf.to_csv("/lakehouse/default/Files/scm_eval_questions.csv", index=False)
print("saved: /lakehouse/default/Files/scm_eval_questions.csv")
qdf.head(10)

## 5. 完了確認

以下のクエリで全テーブルが揃ったか確認します。

In [ ]:
tables = [
    "dim_supplier", "dim_material", "dim_product", "dim_plant", "dim_warehouse", "dim_date",
    "fact_purchase_orders", "fact_goods_receipts", "fact_inventory_snapshot",
    "fact_production_orders", "fact_shipments", "fact_quality_inspections",
    "tbl_001", "tbl_002", "tbl_003", "tbl_004", "tbl_005",
]
for t in tables:
    n = spark.read.table(t).count()
    print(f"{t:35s} {n:>8,} rows")